<a href="https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


1. Grain (What one row means): One row represents one unique content item (content_id / content_hash_id).


2. Table(s) Used: The anonymized starter dataset data/raw/content_refresh_anonymized.csv (supported by dim_content in warehouse workflows).


3. Time Window: Features are computed strictly from historical performance recorded over a prior 90-day observation window (impressions_90d, sessions_90d) prior to the decision point.


4. Target / Proxy Predicted: The outcome being predicted or ranked is content decay, which is defined by the binary proxy label is_declining = (trend_direction == "down").


5. Deliberately Excluded Field(Leakage Rule) & Why: Exclude product decision flags (such as health_score, priority_score, and action_type).
Why: These are rule-based app outputs; including them causes circular learning where the model merely copies existing rules rather than discovering underlying signals from the raw data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I plan to touch is sorted into these four buckets:

1. Feature: Observable search, analytics, and content signals used as inputs for model training (e.g., impressions_90d, sessions_90d, average_position, ctr, content_age_days).

2. Label: The target variable or proxy outcome that my model tries to predict or rank (e.g., is_declining = trend_direction == "down").

3. Context: Join keys, IDs, and grouping metadata used strictly to connect tables or organize outputs, but never fed as direct predictive signals (e.g., content_id, client_id).


4. Excluded: Some fields deliberately left out of model training—such as app product flags (health_score, priority_score). Why: So as to to prevent circular target leakage or privacy risks.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Load starter CSV
df_raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Query 1: Prove Grain (One row really is one unique content item)
unique_ids = df_raw["content_id"].nunique()
total_rows = len(df_raw)
print("=== QUERY 1: GRAIN VERIFICATION ===")
print(f"Total rows: {total_rows:,} | Unique content_ids: {unique_ids:,}")
print(f"✓ Grain Verified (1 row = 1 content item): {total_rows == unique_ids}\n")

# Query 2: Slice Row Count & Date/Age Span
min_age = df_raw["content_age_days"].min()
max_age = df_raw["content_age_days"].max()
print("=== QUERY 2: SLICE ROW COUNT & SPAN ===")
print(f"Raw slice count: {total_rows:,} rows")
print(f"Content age span: {min_age} days to {max_age} days ({max_age/365:.1f} years)\n")

# Query 3: Availability Filtering with IS TRUE
# Filter: impressions_90d > 0 and content_age_days >= 90
availability_condition = (df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)
df_filtered = df_raw[availability_condition].copy()
surviving_rows = len(df_filtered)

print("=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===")
print(f"Filter applied: (impressions_90d > 0) AND (content_age_days >= 90) IS TRUE")
print(f"Surviving rows: {surviving_rows:,} / {total_rows:,} ({surviving_rows/total_rows:.1%})\n")

# Setup target
df_filtered["is_declining"] = df_filtered["trend_direction"] == "down"

# Leakage Trap Experiment
print("=== PART FOUR: LEAKAGE EXPERIMENT ===")

# Honest 5-feature set
honest_features = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "content_age_days"]

X_honest = df_filtered[honest_features].fillna(0)
y = df_filtered["is_declining"]

# Train honest baseline model
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(X_honest, y)
probs_honest = clf_honest.predict_proba(X_honest)[:, 1]

# Evaluate Top-50 Honest Precision
top50_idx_honest = np.argsort(probs_honest)[-50:]
honest_p50 = y.iloc[top50_idx_honest].mean()
print(f"1. Honest Model Precision@50 (5 features) : {honest_p50:.1%}")

# DELIBERATE LEAK: Add a label-derived proxy column on purpose
df_filtered["leaked_trend_proxy"] = (df_filtered["trend_direction"] == "down").astype(int)
leaked_features = honest_features + ["leaked_trend_proxy"]

X_leaked = df_filtered[leaked_features]
clf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
clf_leaked.fit(X_leaked, y)
probs_leaked = clf_leaked.predict_proba(X_leaked)[:, 1]

top50_idx_leaked = np.argsort(probs_leaked)[-50:]
leaked_p50 = y.iloc[top50_idx_leaked].mean()
print(f"2. LEAKED Model Precision@50 (with trap column): {leaked_p50:.1%}  <-- Artificial Perfect Score!")

# REMOVE THE TRAP
df_filtered.drop(columns=["leaked_trend_proxy"], inplace=True)
print("3. ✓ TRAP REMOVED: Deleted leaked_trend_proxy column. RetAINED HONEST SCORE: {:.1%}".format(honest_p50))

=== QUERY 1: GRAIN VERIFICATION ===
Total rows: 30,000 | Unique content_ids: 30,000
✓ Grain Verified (1 row = 1 content item): True

=== QUERY 2: SLICE ROW COUNT & SPAN ===
Raw slice count: 30,000 rows
Content age span: 90 days to 564 days (1.5 years)

=== QUERY 3: AVAILABILITY FILTER (IS TRUE) ===
Filter applied: (impressions_90d > 0) AND (content_age_days >= 90) IS TRUE
Surviving rows: 30,000 / 30,000 (100.0%)

=== PART FOUR: LEAKAGE EXPERIMENT ===
1. Honest Model Precision@50 (5 features) : 100.0%
2. LEAKED Model Precision@50 (with trap column): 100.0%  <-- Artificial Perfect Score!
3. ✓ TRAP REMOVED: Deleted leaked_trend_proxy column. RetAINED HONEST SCORE: 100.0%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset represents a static, anonymized 90-day aggregate snapshot. Because different clients onboarded at different times, historical search tracking forms an unbalanced panel ranging from short windows to 12+ months.
Consequently, macro traffic changes driven by broader market seasonality cannot be easily separated from genuine page-level content decay without multi-year historical depth.
Furthermore, the snapshot provides observational decision-support ranking but cannot prove causal traffic recovery from a refresh without a formal experimental design.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.